# AMEX Enterprise Credit Risk Platform
## Notebook 03 — Data Validation & Exploratory Data Analysis
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Data Understanding + Data Preparation (quality gate)**. Sprint 1, Notebook 3 of 18. Depends on Notebooks 01 and 02 (reads `artifacts/project_config.json` and `artifacts/notebook_02_summary.json`) -- run those first if you have not already.

**What this notebook does:** runs a structural/integrity validation pass over the customer-level feature stores Notebook 02 produced (schema checks, duplicate checks, train/test population disjointness, hard invariant checks, missingness and outlier profiling), then produces the EDA summary: 4 enhanced charts (target distribution, missing-value profile, feature distributions, correlation heatmap) plus a full descriptive-statistics table. Every chart follows this project's validated data-visualization standard (categorical/sequential/diverging color rules, thin marks, recessive gridlines, direct labels) rather than default, unstyled plots.

**Zero-fabrication rule (same as Notebooks 01-02):** every number, chart, and statistic below is computed live by this cell during this run, from the real files Notebook 02 produced. Nothing is carried over from any prior session or hardcoded from memory.

**Run the single code cell below, once.** It is idempotent -- every output file (charts, CSVs, JSON reports) is written to the same fixed path and overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01 & 02
# =============================================================================
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01 & 02")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
WARP_THREAD_COUNT = (
    PROJECT_CONFIG.get("resource_limits", {}).get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)  # 95% of logical cores when Notebook 01's resource_limits block is present;
   # falls back to the raw core count on an older project_config.json.

TRAIN_FULL_PATH = Path(NB02_SUMMARY["output_files"]["train_full_features.parquet"])
TEST_FEATURES_PATH = Path(NB02_SUMMARY["output_files"]["test_features.parquet"])

for _p in (TRAIN_FULL_PATH, TEST_FEATURES_PATH):
    if not _p.exists():
        raise FileNotFoundError(
            f"{_p} not found -- Notebook 02's summary artifact references this path but the "
            "file is missing. Fix: re-run 02_data_engineering.ipynb."
        )

DATA_VALIDATION_DIR = PILLAR_DIRS["data_validation"]
CHARTS_DIR = DATA_VALIDATION_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from     : {CONFIG_PATH}")
print(f"Loaded NB02 summary from: {NB02_SUMMARY_PATH}")
print(f"train_full_features.parquet: {TRAIN_FULL_PATH} ({TRAIN_FULL_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"test_features.parquet      : {TEST_FEATURES_PATH} ({TEST_FEATURES_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"Outputs will be written to : {DATA_VALIDATION_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import matplotlib
    matplotlib.use("Agg")  # headless-safe backend; Jupyter still renders inline via %matplotlib inline / display()
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
except ImportError:
    missing.append("matplotlib")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (WARP 6.4, Concurrency)")
print("\n\u2705 Section 2 complete -- polars, matplotlib, numpy imported.")


# =============================================================================
# SECTION 3: VALIDATED CHART STYLE -- THIS PROJECT'S DATA-VISUALIZATION STANDARD
# =============================================================================
_section("SECTION 3: Validated Chart Style")

# --- Colors below are the validated default palette (categorical hue order,
#     sequential single-hue ramp, diverging blue<->red pair) -- chosen by a
#     computed CVD-safety/contrast validator, not eyeballed. Applied
#     consistently so every chart in this platform reads as one system. ---
VIZ = {
    "surface": "#fcfcfb",
    "text_primary": "#0b0b0b",
    "text_secondary": "#52514e",
    "grid": "#e3e2dd",
    "cat_blue": "#2a78d6",     # categorical slot 1 -- "no default" / primary series
    "cat_red": "#e34948",      # categorical slot 8 -- "default" / risk series
    "seq_blue_mid": "#3987e5",
    "diverging_neutral": "#f0efec",
}


def _style_axes(ax):
    """Recessive grid/axes, consistent surface and text color -- applied to
    every chart in this notebook so they read as one design system."""
    ax.set_facecolor(VIZ["surface"])
    ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"])
    ax.yaxis.label.set_color(VIZ["text_secondary"])


print("VIZ palette and _style_axes() defined -- applied to every chart below (Sections 7-10).")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD AGGREGATED FEATURE STORES (LIVE)
# =============================================================================
_section("SECTION 4: Load Aggregated Feature Stores")

_t0 = time.time()
train_df = pl.read_parquet(str(TRAIN_FULL_PATH))
test_df = pl.read_parquet(str(TEST_FEATURES_PATH))
print(f"Loaded train_full_features.parquet: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns "
      f"({time.time() - _t0:.1f}s)")
print(f"Loaded test_features.parquet      : {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")

# --- Column classification, derived live from the actual column names Notebook
#     02 produced (suffix convention: _mean/_std/_min/_max/_last = numeric,
#     _last/_nunique on a categorical base = categorical), not hardcoded. ---
_all_cols = train_df.columns
_numeric_last_cols = [c for c in _all_cols if c.endswith("_last") and (c[:-5] + "_mean") in _all_cols]
_categorical_last_cols = [c for c in _all_cols if c.endswith("_last") and c not in _numeric_last_cols]
print(f"Numeric '_last' columns (representative per-customer value): {len(_numeric_last_cols)}")
print(f"Categorical '_last' columns: {len(_categorical_last_cols)}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: STRUCTURAL & INTEGRITY VALIDATION
# =============================================================================
_section("SECTION 5: Structural & Integrity Validation")

VALIDATION_CHECKS = []


def _check(name: str, passed: bool, detail: str, severity: str = "error"):
    VALIDATION_CHECKS.append({"check": name, "passed": bool(passed), "detail": detail, "severity": severity})
    status = "\u2705 PASS" if passed else ("\u274c FAIL" if severity == "error" else "\u26a0\ufe0f  WARN")
    print(f"{status} -- {name}: {detail}")


# 1. No duplicate customer_ID in either feature store (impossible after a
#    correct group_by, but a defensive, live check rather than an assumption).
_train_dupes = train_df.shape[0] - train_df["customer_ID"].n_unique()
_test_dupes = test_df.shape[0] - test_df["customer_ID"].n_unique()
_check("No duplicate customer_ID (train)", _train_dupes == 0, f"{_train_dupes} duplicate rows found")
_check("No duplicate customer_ID (test)", _test_dupes == 0, f"{_test_dupes} duplicate rows found")

# 2. Train and test customer populations are disjoint (official AMEX design;
#    verified empirically, not assumed).
_overlap = train_df.select("customer_ID").join(test_df.select("customer_ID"), on="customer_ID", how="inner")
_check("Train/test customer_ID populations are disjoint", _overlap.shape[0] == 0,
       f"{_overlap.shape[0]} customer_ID values found in both populations")

# 3. Hard invariant: a customer can have at most 13 monthly statements
#    (the AMEX observation window), and at least 1 (every customer in these
#    files has some history).
_bad_stmt_count = train_df.filter((pl.col("statement_count") < 1) | (pl.col("statement_count") > 13)).shape[0]
_check("statement_count within [1, 13] for every train customer", _bad_stmt_count == 0,
       f"{_bad_stmt_count} customers outside the valid range")
_bad_stmt_count_test = test_df.filter((pl.col("statement_count") < 1) | (pl.col("statement_count") > 13)).shape[0]
_check("statement_count within [1, 13] for every test customer", _bad_stmt_count_test == 0,
       f"{_bad_stmt_count_test} customers outside the valid range")

# 4. tenure_days must be non-negative and cannot exceed ~13 months.
_bad_tenure = train_df.filter((pl.col("tenure_days") < 0) | (pl.col("tenure_days") > 400)).shape[0]
_check("tenure_days within [0, 400] for every train customer", _bad_tenure == 0,
       f"{_bad_tenure} customers outside the expected range")

# 5. target column is strictly binary (0/1) with no nulls -- this feeds every
#    downstream modeling notebook, so a violation here must block, not warn.
_target_values = set(train_df["target"].unique().to_list())
_check("target column is strictly {0, 1}, no nulls", _target_values <= {0, 1} and train_df["target"].null_count() == 0,
       f"observed values: {sorted(_target_values)}, nulls: {train_df['target'].null_count()}")

_hard_failures = [c for c in VALIDATION_CHECKS if not c["passed"] and c["severity"] == "error"]
if _hard_failures:
    raise RuntimeError(
        f"{len(_hard_failures)} hard data-integrity check(s) failed -- see \u274c lines above. "
        "Fix: investigate Notebook 02's aggregation before proceeding; do not silently continue "
        "past a failed integrity check."
    )

print(f"\nAll {len(VALIDATION_CHECKS)} structural/integrity checks passed.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: MISSING VALUE & OUTLIER PROFILING
# =============================================================================
_section("SECTION 6: Missing Value & Outlier Profiling")

_t0 = time.time()
_missing_counts = train_df.select([pl.col(c).null_count().alias(c) for c in _all_cols]).to_dicts()[0]
_n_rows = train_df.shape[0]
missing_report = sorted(
    [{"column": c, "missing_count": v, "missing_pct": round(100 * v / _n_rows, 3)} for c, v in _missing_counts.items() if v > 0],
    key=lambda r: r["missing_pct"], reverse=True,
)
missing_report_path = DATA_VALIDATION_DIR / "missing_value_report.csv"
pl.DataFrame(missing_report).write_csv(missing_report_path) if missing_report else \
    pl.DataFrame({"column": [], "missing_count": [], "missing_pct": []}).write_csv(missing_report_path)
print(f"Missing-value profile computed over {len(_all_cols)} columns in {time.time() - _t0:.1f}s")
print(f"Columns with any missing values: {len(missing_report)} / {len(_all_cols)}")
if missing_report:
    print("Top 5 most-missing columns:")
    for r in missing_report[:5]:
        print(f"  {r['column']:<28} {r['missing_pct']:>6.2f}% missing ({r['missing_count']:,} customers)")
print(f"\u2705 Saved -> {missing_report_path}")

# --- Outlier profiling: standard Tukey IQR rule (1.5x IQR beyond Q1/Q3),
#     applied to the numeric '_last' columns as the representative per-customer
#     value. This is a REPORTING step -- no rows or values are altered or
#     removed here; that is Notebook 04 (Feature Engineering)'s decision to make. ---
_t0 = time.time()
IQR_MULTIPLIER = 1.5
outlier_rows = []
for c in _numeric_last_cols:
    _q1, _q3 = train_df.select([pl.col(c).quantile(0.25).alias("q1"), pl.col(c).quantile(0.75).alias("q3")]).row(0)
    if _q1 is None or _q3 is None:
        continue
    _iqr = _q3 - _q1
    _lo, _hi = _q1 - IQR_MULTIPLIER * _iqr, _q3 + IQR_MULTIPLIER * _iqr
    _n_out = train_df.filter((pl.col(c) < _lo) | (pl.col(c) > _hi)).shape[0]
    if _n_out > 0:
        outlier_rows.append({"column": c, "q1": round(_q1, 4), "q3": round(_q3, 4),
                              "lower_bound": round(_lo, 4), "upper_bound": round(_hi, 4),
                              "outlier_count": _n_out, "outlier_pct": round(100 * _n_out / _n_rows, 3)})
outlier_rows.sort(key=lambda r: r["outlier_pct"], reverse=True)
outlier_report_path = DATA_VALIDATION_DIR / "outlier_report.csv"
(pl.DataFrame(outlier_rows) if outlier_rows else
 pl.DataFrame({"column": [], "q1": [], "q3": [], "lower_bound": [], "upper_bound": [], "outlier_count": [], "outlier_pct": []})
 ).write_csv(outlier_report_path)
print(f"\nOutlier profile (Tukey IQR x{IQR_MULTIPLIER}) computed over {len(_numeric_last_cols)} numeric columns "
      f"in {time.time() - _t0:.1f}s")
if outlier_rows:
    print("Top 5 columns by outlier share:")
    for r in outlier_rows[:5]:
        print(f"  {r['column']:<28} {r['outlier_pct']:>6.2f}% outliers ({r['outlier_count']:,} customers)")
print(f"\u2705 Saved -> {outlier_report_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: EDA CHART 1 -- TARGET / DEFAULT RATE DISTRIBUTION
# =============================================================================
_section("SECTION 7: EDA Chart 1 -- Target / Default Rate Distribution")

_target_counts = train_df.group_by("target").agg(pl.len().alias("n")).sort("target")
_labels = ["No Default (target=0)", "Default (target=1)"]
_counts = [
    _target_counts.filter(pl.col("target") == 0)["n"][0] if 0 in _target_counts["target"].to_list() else 0,
    _target_counts.filter(pl.col("target") == 1)["n"][0] if 1 in _target_counts["target"].to_list() else 0,
]
_pcts = [100 * c / sum(_counts) for c in _counts]

fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
bars = ax.bar(_labels, _counts, color=[VIZ["cat_blue"], VIZ["cat_red"]], width=0.55, zorder=3)
_style_axes(ax)
ax.set_ylabel("Customers")
ax.set_title(f"Target Distribution -- {sum(_counts):,} Labeled Customers (Live-Computed, This Run)",
             fontsize=11, fontweight="bold", pad=14)
for bar, count, pct in zip(bars, _counts, _pcts):
    ax.annotate(f"{count:,}\n({pct:.2f}%)", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 6), textcoords="offset points", ha="center", va="bottom",
                fontsize=10, color=VIZ["text_primary"], fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
fig.tight_layout()
chart1_path = CHARTS_DIR / "01_target_distribution.png"
fig.savefig(chart1_path, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: EDA CHART 2 -- MISSING VALUE PROFILE (TOP 20)
# =============================================================================
_section("SECTION 8: EDA Chart 2 -- Missing Value Profile")

_top_missing = missing_report[:20]
if _top_missing:
    _cols_ = [r["column"] for r in _top_missing][::-1]
    _pcts_ = [r["missing_pct"] for r in _top_missing][::-1]

    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(_cols_))), dpi=150)
    ax.barh(_cols_, _pcts_, color=VIZ["seq_blue_mid"], zorder=3)
    _style_axes(ax)
    ax.grid(axis="x", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("% Missing (of live-computed customer count, this run)")
    ax.set_title(f"Top {len(_cols_)} Columns by Missing-Value Rate", fontsize=11, fontweight="bold", pad=14)
    for i, (col, pct) in enumerate(zip(_cols_, _pcts_)):
        ax.annotate(f"{pct:.1f}%", xy=(pct, i), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=8, color=VIZ["text_secondary"])
    fig.tight_layout()
    chart2_path = CHARTS_DIR / "02_missing_value_profile.png"
    fig.savefig(chart2_path, facecolor=VIZ["surface"])
    plt.show()
    plt.close(fig)
    print(f"\u2705 Saved -> {chart2_path}")
else:
    chart2_path = None
    print("No missing values found in any column -- chart skipped (nothing to show).")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: EDA CHART 3 -- FEATURE DISTRIBUTION HISTOGRAMS
# =============================================================================
_section("SECTION 9: EDA Chart 3 -- Feature Distribution Histograms")

# --- Representative sample chosen by a live, principled rule (lowest
#     missingness among numeric '_last' columns = most complete data to show),
#     not picked by name/memory. ---
_missing_pct_by_col = {r["column"]: r["missing_pct"] for r in missing_report}
_numeric_by_completeness = sorted(_numeric_last_cols, key=lambda c: _missing_pct_by_col.get(c, 0.0))
_sample_cols = _numeric_by_completeness[:4]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), dpi=150)
for ax, col in zip(axes.flat, _sample_cols):
    _values = train_df[col].drop_nulls().to_numpy()
    ax.hist(_values, bins=40, color=VIZ["cat_blue"], edgecolor=VIZ["surface"], linewidth=0.4, zorder=3)
    _style_axes(ax)
    ax.set_title(col, fontsize=10, fontweight="bold")
    ax.set_ylabel("Customers")
for ax in axes.flat[len(_sample_cols):]:
    ax.set_visible(False)
fig.suptitle("Feature Distributions -- 4 Most-Complete Numeric Features (Live-Selected, This Run)",
             fontsize=11, fontweight="bold", y=1.02)
fig.tight_layout()
chart3_path = CHARTS_DIR / "03_feature_distributions.png"
fig.savefig(chart3_path, facecolor=VIZ["surface"], bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Sampled columns (lowest missingness): {_sample_cols}")
print(f"\u2705 Saved -> {chart3_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: EDA CHART 4 -- CORRELATION HEATMAP (TOP FEATURES VS TARGET)
# =============================================================================
_section("SECTION 10: EDA Chart 4 -- Correlation Heatmap")

_t0 = time.time()
_corr_with_target = train_df.select([
    pl.corr(pl.col(c), pl.col("target")).alias(c) for c in _numeric_last_cols
]).to_dicts()[0]
_corr_with_target = {c: v for c, v in _corr_with_target.items() if v is not None}
_top_corr_cols = sorted(_corr_with_target, key=lambda c: abs(_corr_with_target[c]), reverse=True)[:15]
print(f"Computed correlation-with-target for {len(_numeric_last_cols)} numeric columns in {time.time() - _t0:.1f}s")
print(f"Top {len(_top_corr_cols)} selected for the heatmap (by |correlation with target|).")

_heatmap_cols = _top_corr_cols + ["target"]
_corr_matrix_df = train_df.select(_heatmap_cols).to_pandas().corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
_cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    "amex_diverging", [VIZ["cat_red"], VIZ["diverging_neutral"], VIZ["cat_blue"]]
)
im = ax.imshow(_corr_matrix_df.values, cmap=_cmap, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(_heatmap_cols)))
ax.set_yticks(range(len(_heatmap_cols)))
ax.set_xticklabels(_heatmap_cols, rotation=90, fontsize=8, color=VIZ["text_secondary"])
ax.set_yticklabels(_heatmap_cols, fontsize=8, color=VIZ["text_secondary"])
for i in range(len(_heatmap_cols)):
    for j in range(len(_heatmap_cols)):
        _v = _corr_matrix_df.values[i, j]
        ax.text(j, i, f"{_v:.2f}", ha="center", va="center",
                fontsize=6, color=VIZ["text_primary"] if abs(_v) < 0.6 else "#ffffff")
ax.set_title(f"Correlation Heatmap -- Top {len(_top_corr_cols)} Features by |corr(feature, target)| + target",
             fontsize=11, fontweight="bold", pad=14)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pearson correlation", color=VIZ["text_secondary"])
fig.patch.set_facecolor(VIZ["surface"])
fig.tight_layout()
chart4_path = CHARTS_DIR / "04_correlation_heatmap.png"
fig.savefig(chart4_path, facecolor=VIZ["surface"], bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart4_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: EDA DESCRIPTIVE STATISTICS TABLE
# =============================================================================
_section("SECTION 11: EDA Descriptive Statistics Table")

_desc_rows = []
for c in _numeric_last_cols:
    _stats = train_df.select([
        pl.col(c).mean().alias("mean"), pl.col(c).std().alias("std"),
        pl.col(c).min().alias("min"), pl.col(c).quantile(0.25).alias("q1"),
        pl.col(c).median().alias("median"), pl.col(c).quantile(0.75).alias("q3"),
        pl.col(c).max().alias("max"), pl.col(c).null_count().alias("missing_count"),
    ]).row(0, named=True)
    _stats["column"] = c
    _desc_rows.append(_stats)

desc_stats_path = DATA_VALIDATION_DIR / "eda_summary_statistics.csv"
pl.DataFrame(_desc_rows).select(
    ["column", "mean", "std", "min", "q1", "median", "q3", "max", "missing_count"]
).write_csv(desc_stats_path)
print(f"Descriptive statistics computed for {len(_desc_rows)} numeric columns.")
print(f"\u2705 Saved -> {desc_stats_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: WRITE DATA VALIDATION REPORT (JSON)
# =============================================================================
_section("SECTION 12: Write Data Validation Report")

validation_report = {
    "notebook": "03_data_validation_eda",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_customers": train_df.shape[0],
    "test_customers": test_df.shape[0],
    "total_columns": len(_all_cols),
    "numeric_last_columns": len(_numeric_last_cols),
    "categorical_last_columns": len(_categorical_last_cols),
    "checks": VALIDATION_CHECKS,
    "overall_status": "PASS" if not _hard_failures else "FAIL",
    "columns_with_missing_values": len(missing_report),
    "columns_with_outliers_iqr": len(outlier_rows),
    "top_5_correlated_with_target": {c: round(_corr_with_target[c], 4) for c in _top_corr_cols[:5]},
}
validation_report_path = DATA_VALIDATION_DIR / "data_validation_report.json"
with open(validation_report_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2)
print(f"\u2705 Saved -> {validation_report_path}")
print(f"Overall validation status: {validation_report['overall_status']}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 12B: VERIFY OUTPUTS
# =============================================================================
_section("SECTION 12B: Verify Outputs Were Written Correctly")

_expected_files = [missing_report_path, outlier_report_path, chart1_path, chart3_path, chart4_path,
                    desc_stats_path, validation_report_path]
if chart2_path is not None:
    _expected_files.append(chart2_path)

all_ok = True
for fp in _expected_files:
    if fp.exists() and fp.stat().st_size > 0:
        print(f"\u2705 {fp.name:<32} {fp.stat().st_size / 1e3:>10,.1f} KB")
    else:
        all_ok = False
        print(f"\u274c MISSING OR EMPTY: {fp}")

if not all_ok:
    raise RuntimeError("One or more Notebook 03 output files failed to write. See \u274c lines above.")

print("\nAll Notebook 03 outputs verified present and non-empty.")
print("\n\u2705 Section 12B complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 03 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 13: Write Notebook 03 Summary Artifact")

notebook_03_summary = {
    "notebook": "03_data_validation_eda",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "validation_status": validation_report["overall_status"],
    "checks_passed": sum(1 for c in VALIDATION_CHECKS if c["passed"]),
    "checks_total": len(VALIDATION_CHECKS),
    "charts": {p.name: str(p) for p in _expected_files if p.suffix == ".png"},
    "reports": {p.name: str(p) for p in _expected_files if p.suffix in (".csv", ".json")},
}
nb03_summary_path = ARTIFACTS_DIR / "notebook_03_summary.json"
with open(nb03_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_03_summary, f, indent=2)
print(f"\u2705 Saved -> {nb03_summary_path} (Notebook 17 reads this file to build the rolled-up Data Validation & EDA section)")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 14: Notebook 03 Complete -- Handoff to Notebook 04")

print("NOTEBOOK 03: DATA VALIDATION & EDA -- COMPLETE")
print(f"  Validation status          : {validation_report['overall_status']} "
      f"({notebook_03_summary['checks_passed']}/{notebook_03_summary['checks_total']} checks passed)")
print(f"  Columns with missing values: {len(missing_report)} / {len(_all_cols)}")
print(f"  Columns with IQR outliers  : {len(outlier_rows)}")
print(f"  Charts produced            : {len(notebook_03_summary['charts'])}")
for _name in notebook_03_summary["charts"]:
    print(f"    - {_name}")
print(f"  Reports produced           : {len(notebook_03_summary['reports'])}")
for _name in notebook_03_summary["reports"]:
    print(f"    - {_name}")
print(f"  Next notebook              : 04_feature_engineering.ipynb (Sprint 1)")
print("\n\u2705 Ready to proceed.")
